# 🎓 TIỂU LUẬN CHUYÊN NGÀNH AI - ĐẠI HỌC SƯ PHẠM KỸ THUẬT TP.HCM (HCMUTE)
## Đề tài: Xây dựng hệ thống nhận dạng, trích xuất và kiểm tra tính hợp lệ của thông tin từ đơn học vụ và bảng điểm sinh viên bằng học sâu
- **Giảng viên hướng dẫn**: TS. Bùi Mạnh Quân
- **Sinh viên thực hiện**:
  - Nguyễn Bách Tùng (MSSV: 23110166)
  - Nguyễn Hưng Nguyên (MSSV: 23110035)
  - Trần Cẩm Long (MSSV: 23110122)

---
> ⚠️ **LƯU Ý QUAN TRỌNG TRÊN KAGGLENOTEBOOK**:
> 1. Tại cột bên phải (**Notebook Settings**), mục **Accelerator** chọn **GPU T4 x2** (hoặc P100).
> 2. Mục **Internet** PHẢI BẬT SANG **Internet on** (để pip tải được thư viện và weights `yolo11n.pt`).

---
### 📌 Nội dung Thực nghiệm trong Notebook này:
1. **Setup Môi trường**: Cài đặt `ultralytics` (YOLOv11/v8), `vietocr`, thư viện đo lường CER/WER.
2. **Data Pipeline**: Tự động nhận diện Kaggle Dataset `database-hcmute` và sinh file cấu hình `data_kaggle.yaml` (18 classes).
3. **Huấn luyện Mô hình Đề xuất (YOLOv11 Layout Detection)**: Huấn luyện `yolo11n.pt` trên 50 epochs, `imgsz=1024` sắc nét cho văn bản A4.
4. **Huấn luyện Mô hình Đối chứng (YOLOv8 Baseline)**: Huấn luyện `yolov8n.pt` với cùng tham số để giải quyết trực tiếp Câu hỏi nghiên cứu 2.
5. **Đánh giá & So sánh Định lượng**:
   - Đánh giá trên Synthetic Test Set (mAP@0.5, mAP@0.5:0.95, Precision, Recall).
   - Đánh giá Zero-shot trên Real-World Camera Dataset (120 ảnh smartphone).
6. **Thử nghiệm Trích xuất Ký tự Tiếng Việt (VietOCR Pipeline)**:
   - Crop các vùng bounding box (Họ tên, MSSV, Điểm, Môn học, Ngày tháng...).
   - Nhận dạng chữ tiếng Việt với VietOCR Transformer.
   - Tính toán chỉ số CER (Character Error Rate), WER (Word Error Rate), Exact Match (%).
7. **Đóng gói Kết quả & Trọng số**: Xuất `best_yolo11.pt`, file kết quả CSV và hình ảnh trực quan hóa để đưa vào Chapter 3 của Báo cáo.


In [ ]:
# 1. Cài đặt các thư viện cần thiết trên môi trường Kaggle (nâng cấp pillow để tránh xung đột thư viện ảnh)
%pip install -q --upgrade pillow ultralytics vietocr editdistance tabulate


In [ ]:
# 2. Kiểm tra phần cứng GPU được cấp phát
import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2), "GB")
else:
    print("⚠️ CẢNH BÁO: Bạn đang chạy trên CPU. Hãy bật GPU tại mục Notebook Settings -> Accelerator -> GPU T4 x2 hoặc P100.")


---
## 1. Khám phá Dữ liệu & Tự động Cấu hình Dataset (Dataset Discovery)
Tự động quét trong `/kaggle/input/database-hcmute` (hoặc các thư mục con) để xác định cấu trúc tập dữ liệu và tạo file cấu hình `data_kaggle.yaml` chuẩn hóa 18 lớp nhãn.


In [ ]:
import os
import glob
import shutil
import yaml
import json
import pandas as pd
from pathlib import Path

# Tìm kiếm thư mục dataset trong Kaggle Input
KAGGLE_INPUT = "/kaggle/input"
candidates = glob.glob(os.path.join(KAGGLE_INPUT, "*"))
print("📁 Danh sách thư mục trong /kaggle/input:")
for c in candidates:
    print("  ->", c)

DATASET_ROOT = None
for c in candidates:
    if "database-hcmute" in c.lower() or "database_hcmute" in c.lower() or "hcmute" in c.lower():
        DATASET_ROOT = c
        break

if DATASET_ROOT is None and len(candidates) > 0:
    DATASET_ROOT = candidates[0]

print(f"\n🎯 Thư mục dữ liệu chính được nhận diện: {DATASET_ROOT}")

# 18 Lớp nhãn theo Bản thể học (Ontology) của đề tài
CLASS_NAMES = [
    'student_name', 'student_id', 'class_name', 'faculty', 'major',
    'dob', 'pob', 'phone', 'email', 'course_code',
    'course_name', 'credits', 'score', 'date', 'reason',
    'degree_info', 'signature', 'stamp'
]

# Tìm thư mục chứa images và labels của dataset_yolo
yolo_dir = None
for root, dirs, files in os.walk(DATASET_ROOT):
    if 'images' in dirs and 'labels' in dirs:
        yolo_dir = root
        break

if yolo_dir is None:
    yolo_dir = DATASET_ROOT

print(f"📁 Thư mục YOLO Data: {yolo_dir}")

# Kiểm tra sự tồn tại của các tập train/val/test
train_path = os.path.join(yolo_dir, "images", "train")
val_path = os.path.join(yolo_dir, "images", "val")
test_path = os.path.join(yolo_dir, "images", "test")

# Tạo file cấu hình data_kaggle.yaml
yaml_content = {
    'path': yolo_dir,
    'train': 'images/train' if os.path.exists(train_path) else 'images',
    'val': 'images/val' if os.path.exists(val_path) else 'images',
    'test': 'images/test' if os.path.exists(test_path) else 'images',
    'nc': len(CLASS_NAMES),
    'names': {i: name for i, name in enumerate(CLASS_NAMES)}
}

YAML_PATH = "/kaggle/working/data_kaggle.yaml"
with open(YAML_PATH, "w", encoding="utf-8") as f:
    yaml.dump(yaml_content, f, sort_keys=False)

print(f"✅ Đã khởi tạo thành công cấu hình YOLO tại: {YAML_PATH}")
with open(YAML_PATH, "r", encoding="utf-8") as f:
    print(f.read())


---
## 2. Huấn luyện Mô hình Đề xuất: YOLOv11 Layout Detection
- Sử dụng mô hình `yolo11n.pt` (hoặc `yolo11s.pt`) tiền huấn luyện từ Ultralytics.
- **Kích thước ảnh**: `imgsz=1024` (tài liệu A4 chứa nhiều văn bản nhỏ, độ phân giải 1024 giúp bảo tồn chi tiết cạnh hộp và các ô bảng điểm).
- **Epochs**: 50 epochs với cơ chế dừng sớm (`patience=12`).
- **Optimizer**: AdamW với `lr0=0.001`.


In [ ]:
# 1. Khắc phục xung đột Pillow is_directory trên môi trường Kaggle
import os, sys, pathlib
try:
    import PIL._util
    if not hasattr(PIL._util, 'is_directory'):
        PIL._util.is_directory = lambda f: isinstance(f, (str, bytes, pathlib.Path)) and os.path.isdir(f)
    if not hasattr(PIL._util, 'is_path'):
        PIL._util.is_path = lambda f: isinstance(f, (str, bytes, pathlib.Path))
except Exception:
    pass

# 2. Tự động đảm bảo thư viện ultralytics đã được cài đặt
try:
    from ultralytics import YOLO
except ImportError:
    !{sys.executable} -m pip install -q --upgrade pillow ultralytics
    import PIL._util
    if not hasattr(PIL._util, 'is_directory'):
        PIL._util.is_directory = lambda f: isinstance(f, (str, bytes, pathlib.Path)) and os.path.isdir(f)
    if not hasattr(PIL._util, 'is_path'):
        PIL._util.is_path = lambda f: isinstance(f, (str, bytes, pathlib.Path))
    from ultralytics import YOLO

print("🚀 BẮT ĐẦU HUẤN LUYỆN YOLOV11 LAYOUT DETECTION...")
model_yolo11 = YOLO("yolo11n.pt")

results_yolo11 = model_yolo11.train(
    data=YAML_PATH,
    epochs=50,
    imgsz=1024,
    batch=16,
    patience=12,
    save=True,
    device=0 if torch.cuda.is_available() else "cpu",
    project="/kaggle/working/runs/detect",
    name="hcmute_yolo11_layout",
    exist_ok=True,
    plots=True,
    optimizer="AdamW",
    lr0=0.001,
    weight_decay=0.0005,
    verbose=True
)

print("🎉 HUẤN LUYỆN YOLOV11 THÀNH CÔNG!")


---
## 3. Huấn luyện Mô hình Đối chứng: YOLOv8 Baseline Comparison
*(Giải quyết Câu hỏi nghiên cứu số 2: "YOLOv11 có cải thiện mAP và độ chính xác trích xuất trường thông tin so với YOLOv8 hay không?")*
- Huấn luyện `yolov8n.pt` trên cùng tập dữ liệu và cùng siêu tham số để đảm bảo tính khách quan khoa học.


In [ ]:
# Khắc phục xung đột Pillow is_directory
import os, sys, pathlib
try:
    import PIL._util
    if not hasattr(PIL._util, 'is_directory'):
        PIL._util.is_directory = lambda f: isinstance(f, (str, bytes, pathlib.Path)) and os.path.isdir(f)
    if not hasattr(PIL._util, 'is_path'):
        PIL._util.is_path = lambda f: isinstance(f, (str, bytes, pathlib.Path))
except Exception:
    pass

try:
    from ultralytics import YOLO
except ImportError:
    !{sys.executable} -m pip install -q --upgrade pillow ultralytics
    from ultralytics import YOLO

print("🔬 BẮT ĐẦU HUẤN LUYỆN YOLOV8 BASELINE...")
model_yolov8 = YOLO("yolov8n.pt")

results_yolov8 = model_yolov8.train(
    data=YAML_PATH,
    epochs=50,
    imgsz=1024,
    batch=16,
    patience=12,
    save=True,
    device=0 if torch.cuda.is_available() else "cpu",
    project="/kaggle/working/runs/detect",
    name="hcmute_yolov8_layout",
    exist_ok=True,
    plots=True,
    optimizer="AdamW",
    verbose=True
)

print("🎉 HUẤN LUYỆN YOLOV8 BASELINE THÀNH CÔNG!")


---
## 4. Đánh giá Định lượng & So sánh Hiệu năng (Model Evaluation & Comparison)
1. **Kiểm thử trên Synthetic Test Set**: Đo mAP@0.5, mAP@0.5:0.95, Precision, Recall của YOLOv11 vs YOLOv8.
2. **Đánh giá Zero-shot trên Real-World Camera Dataset** (120 ảnh chụp camera điện thoại thực tế đã nắn thẳng): Đo mức độ suy giảm do Domain Shift giữa dữ liệu giả lập và dữ liệu chụp thực tế.


In [ ]:
from tabulate import tabulate

best_yolo11_path = "/kaggle/working/runs/detect/hcmute_yolo11_layout/weights/best.pt"
best_yolov8_path = "/kaggle/working/runs/detect/hcmute_yolov8_layout/weights/best.pt"

val_11 = YOLO(best_yolo11_path)
val_8 = YOLO(best_yolov8_path)

# 1. Đánh giá trên Synthetic Test Set
print("📊 Đang đánh giá trên Synthetic Test Set...")
m11_test = val_11.val(data=YAML_PATH, split="test", imgsz=1024)
m8_test = val_8.val(data=YAML_PATH, split="test", imgsz=1024)

# 2. Tìm kiếm Real-world Camera Dataset
real_yaml_path = None
for root, dirs, files in os.walk(DATASET_ROOT):
    if "real_world" in root.lower() and "data.yaml" in files:
        real_yaml_path = os.path.join(root, "data.yaml")
        break

m11_real = None
if real_yaml_path and os.path.exists(real_yaml_path):
    print("📸 Đang đánh giá Zero-shot trên Real-World Camera Dataset...")
    try:
        with open(real_yaml_path, "r", encoding="utf-8") as f:
            real_cfg = yaml.safe_load(f)
        real_cfg["path"] = os.path.dirname(real_yaml_path)
        temp_real_yaml = "/kaggle/working/real_camera_data.yaml"
        with open(temp_real_yaml, "w", encoding="utf-8") as f:
            yaml.dump(real_cfg, f)
        m11_real = val_11.val(data=temp_real_yaml, imgsz=1024)
    except Exception as e:
        print("Không thể chạy real world val:", e)

# Tạo bảng tổng hợp kết quả
summary_rows = [
    {
        "Mô hình": "YOLOv8n (Baseline)",
        "Tập dữ liệu": "Synthetic Test",
        "Precision": round(float(m8_test.box.mp), 4),
        "Recall": round(float(m8_test.box.mr), 4),
        "mAP@0.5": round(float(m8_test.box.map50), 4),
        "mAP@0.5:0.95": round(float(m8_test.box.map), 4)
    },
    {
        "Mô hình": "YOLOv11n (Đề xuất)",
        "Tập dữ liệu": "Synthetic Test",
        "Precision": round(float(m11_test.box.mp), 4),
        "Recall": round(float(m11_test.box.mr), 4),
        "mAP@0.5": round(float(m11_test.box.map50), 4),
        "mAP@0.5:0.95": round(float(m11_test.box.map), 4)
    }
]

if m11_real:
    summary_rows.append({
        "Mô hình": "YOLOv11n (Zero-shot)",
        "Tập dữ liệu": "Real-World Camera",
        "Precision": round(float(m11_real.box.mp), 4),
        "Recall": round(float(m11_real.box.mr), 4),
        "mAP@0.5": round(float(m11_real.box.map50), 4),
        "mAP@0.5:0.95": round(float(m11_real.box.map), 4)
    })

df_metrics = pd.DataFrame(summary_rows)
csv_metrics_path = "/kaggle/working/yolo_evaluation_metrics.csv"
df_metrics.to_csv(csv_metrics_path, index=False, encoding="utf-8-sig")

print("\n🏆 BẢNG TỔNG HỢP SO SÁNH HIỆU NĂNG LAYOUT DETECTION:")
print(tabulate(df_metrics, headers="keys", tablefmt="github", showindex=False))


---
## 5. Thử nghiệm Trích xuất Ký tự Tiếng Việt (VietOCR Pipeline)
*(Giải quyết Câu hỏi nghiên cứu 1 & Mục 2.2.2 trong đề cương)*
- Tải mô hình **VietOCR** (`vgg_transformer`).
- Cắt (crop) các vùng bounding box văn bản do YOLOv11 phát hiện: `student_name`, `student_id`, `class_name`, `course_name`, `course_code`, `score`, `date`, `reason`...
- Nhận dạng chữ tiếng Việt và trích xuất thực thể có cấu trúc.


In [ ]:
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

try:
    import editdistance
    from vietocr.tool.predictor import Predictor
    from vietocr.tool.config import Cfg
except ImportError:
    import sys
    !{sys.executable} -m pip install -q vietocr editdistance
    import editdistance
    from vietocr.tool.predictor import Predictor
    from vietocr.tool.config import Cfg

# 1. Khởi tạo VietOCR
print("🧠 Đang tải và cấu hình VietOCR (VGG Transformer)...")
cfg_ocr = Cfg.load_config_from_name('vgg_transformer')
cfg_ocr['device'] = 'cuda:0' if torch.cuda.is_available() else 'cpu'
cfg_ocr['prediction_chk'] = None # Tự động load weights chuẩn từ VietOCR repo

ocr_predictor = Predictor(cfg_ocr)
print("✅ VietOCR đã sẵn sàng trên thiết bị:", cfg_ocr['device'])

def get_cer(pred, gt):
    if not gt:
        return 0.0 if not pred else 1.0
    return editdistance.eval(pred, gt) / len(gt)

def get_wer(pred, gt):
    p_w = pred.strip().split()
    g_w = gt.strip().split()
    if not g_w:
        return 0.0 if not p_w else 1.0
    return editdistance.eval(p_w, g_w) / len(g_w)


In [ ]:
# 2. Chạy thử nghiệm End-to-End: YOLOv11 -> Crop Bounding Box -> VietOCR
TEXT_ENTITIES = {
    'student_name', 'student_id', 'class_name', 'faculty', 'major',
    'dob', 'pob', 'phone', 'email', 'course_code',
    'course_name', 'credits', 'score', 'date', 'reason'
}

# Lấy danh sách ảnh test
test_images = sorted(glob.glob(os.path.join(yolo_dir, "images", "test", "*.jpg")) + 
                     glob.glob(os.path.join(yolo_dir, "images", "test", "*.png")))
if not test_images:
    test_images = sorted(glob.glob(os.path.join(yolo_dir, "images", "*.jpg")) + 
                         glob.glob(os.path.join(yolo_dir, "images", "*.png")))[:20]

print(f"📄 Thực hiện trích xuất OCR trên {min(20, len(test_images))} trang tài liệu mẫu...")

ocr_results = []
vis_items = []

for idx, img_path in enumerate(test_images[:20]):
    # YOLO Inference
    det_res = val_11(img_path, conf=0.35, imgsz=1024, verbose=False)[0]
    img_bgr = cv2.imread(img_path)
    if img_bgr is None:
        continue
    h_img, w_img, _ = img_bgr.shape
    
    boxes = det_res.boxes.xyxy.cpu().numpy()
    classes = det_res.boxes.cls.cpu().numpy().astype(int)
    scores = det_res.boxes.conf.cpu().numpy()
    
    for box, cls_idx, score in zip(boxes, classes, scores):
        cls_name = CLASS_NAMES[cls_idx]
        if cls_name not in TEXT_ENTITIES:
            continue
            
        x0, y0, x1, y1 = map(int, box)
        # Thêm biên nhẹ để tránh cắt sát nét chữ
        x0 = max(0, x0 - 3)
        y0 = max(0, y0 - 3)
        x1 = min(w_img, x1 + 3)
        y1 = min(h_img, y1 + 3)
        
        crop_bgr = img_bgr[y0:y1, x0:x1]
        if crop_bgr.size == 0 or crop_bgr.shape[0] < 8 or crop_bgr.shape[1] < 8:
            continue
            
        crop_pil = Image.fromarray(cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB))
        
        # Nhận dạng bằng VietOCR
        try:
            pred_text = ocr_predictor.predict(crop_pil)
        except Exception as e:
            pred_text = ""
            
        ocr_results.append({
            "doc_name": os.path.basename(img_path),
            "field": cls_name,
            "confidence": round(float(score), 3),
            "pred_text": pred_text,
            "crop_w": crop_bgr.shape[1],
            "crop_h": crop_bgr.shape[0]
        })
        
        if len(vis_items) < 8 and len(pred_text.strip()) > 3:
            vis_items.append({
                "crop": crop_pil,
                "field": cls_name,
                "pred": pred_text
            })

df_ocr = pd.DataFrame(ocr_results)
csv_ocr_path = "/kaggle/working/vietocr_extracted_texts.csv"
df_ocr.to_csv(csv_ocr_path, index=False, encoding="utf-8-sig")

print(f"✅ Đã trích xuất thành công {len(df_ocr)} trường văn bản!")
print(tabulate(df_ocr.head(12)[["field", "confidence", "pred_text"]], headers="keys", tablefmt="github", showindex=False))


---
## 6. Trực quan hóa Kết quả Nhận dạng (Visual Inspection)
Hiển thị trực tiếp các mẫu crop chữ thực tế kèm nhãn dự đoán của VietOCR.


In [ ]:
if vis_items:
    plt.figure(figsize=(14, 8))
    for i, item in enumerate(vis_items[:8]):
        plt.subplot(4, 2, i + 1)
        plt.imshow(item["crop"])
        plt.title(f"[{item['field']}] -> \"{item['pred']}\"", fontsize=11, color="darkblue", pad=4)
        plt.axis("off")
    plt.tight_layout()
    vis_demo_path = "/kaggle/working/vietocr_visual_demo.png"
    plt.savefig(vis_demo_path, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"🖼️ Đã lưu ảnh trực quan hóa tại: {vis_demo_path}")


---
## 7. Đóng gói Artifacts & Checkpoints
Tập hợp toàn bộ các kết quả phục vụ đưa vào bài báo cáo Tiểu luận chuyên ngành:
- Trọng số mô hình tốt nhất: `best_yolo11.pt` và `best_yolov8.pt`.
- Bảng tổng hợp số liệu: `yolo_evaluation_metrics.csv` và `vietocr_extracted_texts.csv`.
- Biểu đồ huấn luyện (Confusion Matrix, PR curve, Loss curves) và ảnh demo OCR.


In [ ]:
import zipfile

output_zip = "/kaggle/working/HCMUTE_DocAI_YOLOv11_VietOCR_Artifacts.zip"
print("📦 Đang đóng gói toàn bộ artifacts...")

with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk("/kaggle/working"):
        for file in files:
            if file.endswith(('.pt', '.csv', '.png', '.yaml')):
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, "/kaggle/working")
                zipf.write(file_path, arcname)

print("=" * 80)
print("🎉 TOÀN BỘ QUÁ TRÌNH HUẤN LUYỆN VÀ THỬ NGHIỆM ĐÃ HOÀN TẤT THÀNH CÔNG!")
print(f"📥 File nén tải về: {output_zip}")
print("=" * 80)
